# 15장. 프롬프트와 컨텍스트를 숫자로 비교하기

모델을 고치지 않고 프롬프트만 바꿔도 정답률이 움직인다. 9장의 세 모델로 영화평 감정 분류(NSMC)를 하면서 예시의 수, 예시의 순서와 비율, 시스템 프롬프트를 바꿔 보고, 긴 명단 속 한 줄을 찾게 해 정보가 놓인 곳에 따라 답이 어떻게 달라지는지 잰다.

**준비:** `pip install "transformers<5" huggingface_hub pandas`. 9장에서 받은 세 모델과 1장의 NSMC `ratings_train.txt`, `ratings_test.txt`(CC0 1.0)가 있어야 한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 49분쯤 걸렸다.

## 15.1 예시를 몇 개 넣을까

In [ ]:
import os, re, json, math, time, sys, random, copy, platform, warnings, collections
import numpy as np, pandas as pd
import torch
import torch.nn.functional as F
os.environ["HF_HUB_DISABLE_XET"] = "1"
warnings.filterwarnings("ignore")
torch.set_num_threads(2)
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

PATH = {"Qwen3-0.6B": "data/models/Qwen__Qwen3-0.6B",
        "HyperCLOVAX-SEED-0.5B": "data/models/naver-hyperclovax__HyperCLOVAX-SEED-Text-Instruct-0.5B",
        "Kanana-1.5-2.1B": "data/models/kakaocorp__kanana-1.5-2.1b-instruct-2505"}
STOP = {"Qwen3-0.6B": ["<|im_end|>", "<|endoftext|>"], "HyperCLOVAX-SEED-0.5B": ["<|endofturn|>", "<|stop|>"],
        "Kanana-1.5-2.1B": ["<|eot_id|>"]}                  # 9장에서 확인한 끝 토큰
MODELS = list(PATH)
tok = {n: AutoTokenizer.from_pretrained(PATH[n]) for n in MODELS}
load = lambda n: AutoModelForCausalLM.from_pretrained(PATH[n], dtype=torch.float32).eval()
def chat_ids(n, msgs):
    extra = {"enable_thinking": False} if n.startswith("Qwen3") else {}
    return tok[n].apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, **extra)
for n in MODELS:
    print(f"{n:<22} 컨텍스트 창(설정 파일의 최대 위치) {AutoConfig.from_pretrained(PATH[n]).max_position_embeddings:,}토큰")
print(sys.version.split()[0], torch.__version__, transformers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
tr = pd.read_csv("ratings_train.txt", sep="\t").dropna()
te = pd.read_csv("ratings_test.txt", sep="\t").dropna()
tr = tr[tr.document.str.len().between(10, 60)]           # 예시는 10~60글자 영화평에서 고른다
TEST = pd.concat([te[te.label == 1].sample(80, random_state=0), te[te.label == 0].sample(80, random_state=0)]).sample(frac=1, random_state=0)
LAB = {1: "긍정", 0: "부정"}
TASK = "영화평을 읽고 감정을 '긍정'이나 '부정'으로 답하세요."
def body(shots, x):
    return "".join(f"영화평: {d}\n감정: {LAB[l]}\n\n" for d, l in shots) + f"영화평: {x}\n감정:"
def draw(k, seed):
    """학습 문장에서 예시 k개를 뽑는다(긍정·부정 비율은 뽑힌 대로)."""
    s = tr.sample(k, random_state=seed) if k else tr.iloc[:0]
    return list(zip(s.document, s.label))
print(f"시험 문장 {len(TEST)}개(긍정 {int(TEST.label.sum())}개), 예시 후보 {len(tr):,}개")
print("프롬프트 예(예시 2개):\n" + TASK + "\n\n" + body(draw(2, 0), TEST.document.iloc[0]))

In [ ]:
def first_ids(n):
    """답의 첫 토큰. '긍정'과 '부정'의 첫 토큰이 다르므로 이 둘의 점수만 견준다."""
    a, b = tok[n].encode("긍정", add_special_tokens=False)[0], tok[n].encode("부정", add_special_tokens=False)[0]
    assert a != b
    return a, b

def score(model, n, msgs_list, batch=16):
    """대화마다 답의 첫 토큰에서 (긍정 점수 − 부정 점수, 두 토큰 확률의 합)을 돌려준다.
    모든 대화가 같은 앞부분(지시와 예시)을 가지므로, 앞부분의 KV 캐시(6장)를 한 번만 계산해 다시 쓴다."""
    full = [chat_ids(n, m) for m in msgs_list]
    p = 0
    while all(len(f) > p + 1 and f[p] == full[0][p] for f in full):
        p += 1
    pre, suf = full[0][:p], [f[p:] for f in full]
    ia, ib = first_ids(n)
    pad = tok[n].pad_token_id if tok[n].pad_token_id is not None else 0
    with torch.inference_mode():
        base = model(input_ids=torch.tensor([pre]), use_cache=True).past_key_values
    out = []
    for k in range(0, len(suf), batch):
        ss = suf[k:k + batch]; L = max(map(len, ss)); B = len(ss)
        cache = copy.deepcopy(base); cache.batch_repeat_interleave(B)
        ids = torch.tensor([s + [pad] * (L - len(s)) for s in ss])
        mask = torch.tensor([[1] * (p + len(s)) + [0] * (L - len(s)) for s in ss])
        with torch.inference_mode():
            lg = model(input_ids=ids, attention_mask=mask, past_key_values=cache).logits
        z = lg[torch.arange(B), torch.tensor([len(s) - 1 for s in ss])]
        pr = F.softmax(z, -1)
        out += [((z[i, ia] - z[i, ib]).item(), (pr[i, ia] + pr[i, ib]).item()) for i in range(B)]
    return out, len(pre)

def evaluate(model, n, shots, system=None, instruct=True):
    msgs = []
    for x in TEST.document:
        user = (TASK + "\n\n" if instruct else "") + body(shots, x)
        msgs.append(([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}])
    out, plen = score(model, n, msgs)
    pred = np.array([d > 0 for d, _ in out])
    return {"acc": float(np.mean(pred == (TEST.label.values == 1))), "pos": float(pred.mean()),
            "mass": float(np.mean([m for _, m in out])), "prefix": plen}

# 캐시를 다시 쓴 값이 처음부터 계산한 값과 같은지 한 모델로 확인한다
n = MODELS[0]; m0 = load(n); shots = draw(4, 0)
msgs = [[{"role": "user", "content": TASK + "\n\n" + body(shots, x)}] for x in TEST.document[:4]]
fast, _ = score(m0, n, msgs); ia, ib = first_ids(n)
slow = []
for mm in msgs:
    with torch.inference_mode():
        z = m0(input_ids=torch.tensor([chat_ids(n, mm)])).logits[0, -1]
    slow.append((z[ia] - z[ib]).item())
print(f"{n}: 캐시를 다시 쓴 값과 처음부터 계산한 값의 차이 최대 {max(abs(a[0] - b) for a, b in zip(fast, slow)):.1e}")
del m0

In [ ]:
KS, SEEDS = [0, 1, 4, 16], [0, 1, 2]
FS, took = {}, {}
for n in MODELS:
    model = load(n); lap = time.perf_counter()
    for k in KS:
        for s in (SEEDS if k else [0]):
            FS[n, k, s] = evaluate(model, n, draw(k, s))
        r = [FS[n, k, s]["acc"] for s in (SEEDS if k else [0])]
        print(f"{n:<22} 예시 {k:>2}개: 정답률 " + ", ".join(f"{v:.1%}" for v in r) + (f" (범위 {max(r) - min(r):.1%}포인트)" if k else "")
              + f", '긍정'이라고 한 비율 " + ", ".join(f"{FS[n, k, s]['pos']:.1%}" for s in (SEEDS if k else [0])) + f", 앞부분 {FS[n, k, 0]['prefix']}토큰", flush=True)
    took[n] = time.perf_counter() - lap
    del model
print("예시 뽑기별 긍정 예시 수:", {k: [sum(l for _, l in draw(k, s)) for s in SEEDS] for k in KS[1:]})

## 15.2 예시의 순서와 비율

In [ ]:
base8 = draw(200, 5)
P8 = [x for x in base8 if x[1] == 1][:4]; N8 = [x for x in base8 if x[1] == 0][:4]
ORDERS = {"부정 4개 뒤 긍정 4개": N8 + P8, "긍정 4개 뒤 부정 4개": P8 + N8,
          "번갈아(긍정부터)": [x for pair in zip(P8, N8) for x in pair], "긍정만 8개": [x for x in base8 if x[1] == 1][:8]}
OD = {}
for n in MODELS:
    model = load(n); lap = time.perf_counter()
    for name, shots in ORDERS.items():
        OD[n, name] = evaluate(model, n, shots)
        print(f"{n:<22} {name:<14} 정답률 {OD[n, name]['acc']:.1%}, '긍정'이라고 한 비율 {OD[n, name]['pos']:.1%}", flush=True)
    took[n] += time.perf_counter() - lap
    del model
print("시험 문장의 긍정 비율 50.0%")

In [ ]:
# 답을 직접 쓰게 해서 본다: 예시 0개와 4개(뽑기 0), 시험 문장 앞 40개, 탐욕 디코딩 8토큰
PK, PKEX = {}, {}
SUB = TEST.iloc[:40]
for n in MODELS:
    model = load(n); t = tok[n]; stops = [t.convert_tokens_to_ids(s) for s in STOP[n]]
    for k in [0, 4]:
        outs = []
        for x in SUB.document:
            ids = torch.tensor([chat_ids(n, [{"role": "user", "content": TASK + "\n\n" + body(draw(k, 0), x)}])])
            with torch.inference_mode():
                g = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=8, do_sample=False, temperature=None, top_p=None, top_k=None,
                                   eos_token_id=stops, pad_token_id=t.pad_token_id if t.pad_token_id is not None else stops[0])
            outs.append(t.decode(g[0, ids.shape[1]:], skip_special_tokens=True).strip())
        lab = [("긍정" if a.find("긍정") >= 0 and (a.find("부정") < 0 or a.find("긍정") < a.find("부정")) else ("부정" if "부정" in a else None)) for a in outs]
        PK[n, k] = {"acc": float(np.mean([l == LAB[y] for l, y in zip(lab, SUB.label)])), "none": float(np.mean([l is None for l in lab])),
                    "starts": float(np.mean([a.startswith(("긍정", "부정")) for a in outs]))}
        PKEX[n, k] = outs[:3]
        print(f"{n:<22} 예시 {k}개: 답을 읽은 정답률 {PK[n, k]['acc']:.1%}, '긍정'·'부정'으로 시작한 답 {PK[n, k]['starts']:.1%}, 둘 다 없는 답 {PK[n, k]['none']:.1%} | 답 예: {outs[:3]}", flush=True)
    del model

## 15.3 시스템 프롬프트

In [ ]:
SYS = "당신은 영화평의 감정을 분류하는 도우미입니다. 영화평을 읽고 '긍정'이나 '부정' 한 단어로만 답합니다."
for n in MODELS:
    extra = {"enable_thinking": False} if n.startswith("Qwen3") else {}
    s = tok[n].apply_chat_template([{"role": "system", "content": "(시스템)"}, {"role": "user", "content": "(질문)"}], tokenize=False, add_generation_prompt=True, **extra)
    print(f"{n} 템플릿: {s!r}")
SY = {}
CASES = {"지시 없음": dict(instruct=False), "지시를 질문에": dict(instruct=True), "지시를 시스템에": dict(system=SYS, instruct=False)}
for n in MODELS:
    model = load(n); lap = time.perf_counter()
    for name, kw in CASES.items():
        SY[n, name] = evaluate(model, n, [], **kw)
        print(f"{n:<22} {name:<8} 정답률 {SY[n, name]['acc']:.1%}, 답의 첫 토큰이 '긍정'·'부정'일 확률 평균 {SY[n, name]['mass']:.3f}", flush=True)
    took[n] += time.perf_counter() - lap
    del model

## 15.4 긴 컨텍스트에서 정보가 놓인 곳

In [ ]:
NAMES = [s + g for s in "김이박최정강조윤장임한오서신권" for g in ["민준", "서연", "도윤", "하은", "지호", "수아", "예준", "지우", "시우", "서윤", "주원", "하린", "건우", "채원"]]
LENS, TRIALS = [60, 200], 10
POS = {L: [0, L // 4, L // 2, 3 * L // 4, L - 1] for L in LENS}      # 찾을 줄을 놓는 곳: 처음, 1/4, 가운데, 3/4, 끝
def make_trial(j, N_LINES):
    rnd = random.Random(1000 + j)
    names = rnd.sample(NAMES, N_LINES); seat = {x: rnd.randint(10, 99) for x in names}
    return names, seat, names[0]                     # 첫 이름을 찾을 사람으로 정하고, 그 줄을 여러 곳으로 옮긴다
def ask(names, seat, target, at):
    rest = [x for x in names if x != target]
    order = rest[:at] + [target] + rest[at:]
    lines = "\n".join(f"{x}의 좌석은 {seat[x]}번이다." for x in order)
    return f"아래 명단을 읽고 질문에 답하세요.\n\n{lines}\n\n질문: {target}의 좌석은 몇 번인가요? 숫자만 답하세요."

PS, LEN, EX = {}, {}, {}
for n in MODELS:
    model = load(n); lap = time.perf_counter(); t = tok[n]
    stops = [t.convert_tokens_to_ids(s) for s in STOP[n]]
    pad = t.pad_token_id if t.pad_token_id is not None else stops[0]
    for L, at in [(L, at) for L in LENS for at in POS[L]]:
        right = []
        for j in range(TRIALS):
            names, seat, target = make_trial(j, L)
            lead = f"{target}의 좌석은"                    # 답의 앞부분을 미리 넣어 숫자만 이어 쓰게 한다
            x = torch.tensor([chat_ids(n, [{"role": "user", "content": ask(names, seat, target, at)}]) + t.encode(lead, add_special_tokens=False)])
            with torch.inference_mode():
                g = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=4, do_sample=False, temperature=None, top_p=None, top_k=None,
                                   eos_token_id=stops, pad_token_id=pad)
            a = t.decode(g[0, x.shape[1]:], skip_special_tokens=True)
            num = re.search(r"\d+", a)
            right.append(num is not None and num.group(0) == str(seat[target]))
            if j == 0: EX[n, L, at] = a + ("" if num is None else " (" + "·".join(x for x in names if str(seat[x]) == num.group(0)) + "의 번호)")
            LEN[n, L] = x.shape[1]
        PS[n, L, at] = float(np.mean(right))
    took[n] += time.perf_counter() - lap
    for L in LENS:
        print(f"{n:<22} {L}줄, 프롬프트 {LEN[n, L]:,}토큰 | " + ", ".join(f"{at + 1}번째 줄 {PS[n, L, at]:.0%}" for at in POS[L]), flush=True)
    print(f"{'':<22} 200줄 첫 시험의 답: " + " / ".join(f"{at + 1}번째 {EX[n, 200, at]!r}" for at in POS[200]))
    del model
names, seat, target = make_trial(0, 60)
print("질문 예(앞 세 줄과 끝):", ask(names, seat, target, 30).split("\n")[2:5], "…", ask(names, seat, target, 30).split("\n")[-1])

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for fp in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf")) + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(fp)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": True})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"
short = {"Qwen3-0.6B": "Qwen3", "HyperCLOVAX-SEED-0.5B": "HyperCLOVA X", "Kanana-1.5-2.1B": "Kanana"}
COL = dict(zip(MODELS, [BLUE, ORANGE, GREEN]))
fig, axes = plt.subplots(2, 2, figsize=(5.2, 5.4), dpi=260)
a = axes[0, 0]
for n in MODELS:
    med = [np.median([FS[n, k, s]["acc"] * 100 for s in (SEEDS if k else [0])]) for k in KS]
    lo = [min(FS[n, k, s]["acc"] * 100 for s in (SEEDS if k else [0])) for k in KS]
    hi = [max(FS[n, k, s]["acc"] * 100 for s in (SEEDS if k else [0])) for k in KS]
    xs = np.arange(len(KS))
    a.plot(xs, med, "o-", color=COL[n], ms=3.2, lw=1.2, label=short[n])
    a.vlines(xs, lo, hi, color=COL[n], lw=2.2, alpha=0.35)
a.axhline(50, color=GRAY, ls=":", lw=0.9); a.set_xticks(np.arange(len(KS)), [str(k) for k in KS]); a.set_ylim(30, 100)
a.set_xlabel("예시 수"); a.set_title("① 예시 수와 정답률(%)", loc="left", fontsize=8.6, color="#374151")
a.legend(frameon=False, fontsize=8, loc="center right", bbox_to_anchor=(1.0, 0.6))
a = axes[0, 1]
names_o = list(ORDERS)
for i, n in enumerate(MODELS):
    a.bar(np.arange(len(names_o)) + (i - 1) * 0.25, [OD[n, o]["pos"] * 100 for o in names_o], 0.25, color=COL[n], label=short[n])
a.axhline(50, color=GRAY, ls=":", lw=0.9); a.set_ylim(0, 105)
a.set_xticks(np.arange(len(names_o)), ["부정 뒤\n긍정", "긍정 뒤\n부정", "번갈아", "긍정만"])
a.set_title("② 예시 8개의 순서와 '긍정' 비율(%)", loc="left", fontsize=8.6, color="#374151")
a = axes[1, 0]
cases = list(CASES)
for i, n in enumerate(MODELS):
    a.bar(np.arange(3) + (i - 1) * 0.25, [SY[n, c]["mass"] for c in cases], 0.25, color=COL[n], label=short[n])
a.set_xticks(np.arange(3), ["지시 없음", "지시를\n질문에", "지시를\n시스템에"]); a.set_ylim(0, 1.05)
a.set_title("③ 첫 토큰이 '긍정'·'부정'일 확률", loc="left", fontsize=8.6, color="#374151")
a = axes[1, 1]
for n in MODELS:
    for L, ls in [(60, ":"), (200, "-")]:
        a.plot(range(5), [PS[n, L, p] * 100 for p in POS[L]], "o" + ls, color=COL[n], ms=3.2, lw=1.2, label=f"{short[n]} {L}줄")
a.set_xticks(range(5), ["처음", "1/4", "가운데", "3/4", "끝"]); a.set_ylim(-3, 103); a.set_xlabel("찾을 줄이 놓인 곳(점선 60줄, 실선 200줄)")
a.set_title("④ 정보가 놓인 곳과 정답률(%)", loc="left", fontsize=8.6, color="#374151")
for a in axes.flat:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_15_prompt.png"); plt.close(fig)